### FAISS (Facebook AI Similarity Search)

FAISS is an open-source C++ library developed by Meta AI for fast dense vector similarity search and clustering.

It provides highly optimized in-memory indexing with support for saving/loading local index files.

### Key Features

* High performance and low memory footprint.
* Save index to disk (db.save_local(folder_path)).
* Load index from disk (FAISS.load_local(folder_path, embeddings, allow_dangerous_deserialization=True)).
* Merge multiple FAISS indices (db1.merge_from(db2)).
* Convert directly to LangChain Retriever (db.as_retriever()).

In [ ]:
import os
import shutil
from dotenv import load_dotenv, find_dotenv
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
from langchain_community.vectorstores import FAISS
from langchain_google_genai import GoogleGenerativeAIEmbeddings
# from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document

load_dotenv(find_dotenv())

# Initialize Google Generative AI embeddings (commented out HuggingFaceEmbeddings)
# embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
embeddings = GoogleGenerativeAIEmbeddings(model="gemini-embedding-001")

docs_a = [
    Document(page_content="FAISS evaluates vector distances in C++ for maximum throughput."),
    Document(page_content="Inverted file indexes partition search space into Voronoi cells.")
]
docs_b = [
    Document(page_content="HNSW graphs provide sub-linear search time for high dimensional vectors.")
]

# Initialize FAISS vector store
db_a = FAISS.from_documents(docs_a, embeddings)
db_b = FAISS.from_documents(docs_b, embeddings)

# Merge db_b into db_a
db_a.merge_from(db_b)
print("Merged FAISS index contains total document count.")

# Save index locally
index_path = "./faiss_index_demo"
db_a.save_local(index_path)
print(f"Saved FAISS index to {index_path}")

# Re-load index from disk
loaded_db = FAISS.load_local(index_path, embeddings, allow_dangerous_deserialization=True)

# Similarity Search
results = loaded_db.similarity_search("How does HNSW graph search work?", k=1)
print(f"Loaded FAISS Search Result: {results[0].page_content}")

# Convert FAISS to Retriever
retriever = loaded_db.as_retriever(search_kwargs={"k": 1})
retrieved_docs = retriever.invoke("Voronoi cells search")
print(f"Retrieved via Retriever interface: {retrieved_docs[0].page_content}")

if os.path.exists(index_path):
    shutil.rmtree(index_path)


/var/folders/0h/sdy0_vy9385bh841gfp_jzdm0000gn/T/ipykernel_43708/3403853107.py:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS
/Users/kapilyadav/Coding_Space/Python_workspace/LangChainWorkspace/generativeai/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Merged FAISS index contains total document count.
Saved FAISS index to ./faiss_index_demo
Loaded FAISS Search Result: HNSW graphs provide sub-linear search time for high dimensional vectors.
Retrieved via Retriever interface: Inverted file indexes partition search space into Voronoi cells.


: 